# LSTM — Prévision de production éolienne (24h ahead)

Ce notebook entraîne un modèle LSTM pour prédire le taux d'utilisation (`utilization_rate`) 24h à l'avance, en exploitant la nature séquentielle des données météo et de production.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

## 2. Chargement et preprocessing des données

In [ ]:
df = pd.read_csv("../data/cleaned_dataset.csv", index_col=0)
df["delivery_time"] = pd.to_datetime(df["delivery_time"])
df = df.sort_values(["site_name", "delivery_time"]).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Sites: {df['site_name'].nunique()} — {sorted(df['site_name'].unique())}")
print(f"Période: {df['delivery_time'].min()} → {df['delivery_time'].max()}")

In [ ]:
# Colonnes features (on exclut les colonnes meta et la cible)
EXCLUDE_COLS = [
    'site_name', 'delivery_time', 'production',
    'utilization_rate', 'to_drop_for_training',
    'is_maintenance', 'is_confirmed_maint', 'is_curtailment'
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]
TARGET_COL   = 'utilization_rate'
HORIZON      = 24   # prédit 24h en avance
SEQ_LEN      = 48   # fenêtre d'observation (48h de passé)

print(f"{len(FEATURE_COLS)} features: {FEATURE_COLS}")

## 3. Construction des séquences LSTM

Pour chaque pas de temps `t`, on crée :
- **X** : fenêtre `[t − SEQ_LEN, …, t−1]` → shape `(SEQ_LEN, n_features)`
- **y** : `utilization_rate[t + HORIZON]`

On traite chaque site séparément pour éviter de mélanger des séquences inter-sites.

In [ ]:
def make_sequences(site_df, feature_cols, target_col, seq_len, horizon):
    """
    Construit les séquences (X, y) pour un site donné.
    Retourne aussi les timestamps correspondant à la prédiction.
    """
    site_df = site_df.sort_values('delivery_time').reset_index(drop=True)
    values  = site_df[feature_cols].values.astype(np.float32)
    target  = site_df[target_col].values.astype(np.float32)
    times   = site_df['delivery_time'].values

    X, y, ts = [], [], []
    for i in range(seq_len, len(site_df) - horizon):
        X.append(values[i - seq_len: i])         # fenêtre passé
        y.append(target[i + horizon])             # cible future
        ts.append(times[i + horizon])

    return np.array(X), np.array(y), np.array(ts)


all_X, all_y, all_ts, all_sites = [], [], [], []

for site, grp in df.groupby('site_name'):
    X_s, y_s, ts_s = make_sequences(grp, FEATURE_COLS, TARGET_COL, SEQ_LEN, HORIZON)
    all_X.append(X_s)
    all_y.append(y_s)
    all_ts.append(ts_s)
    all_sites.extend([site] * len(y_s))

all_X     = np.concatenate(all_X,  axis=0)
all_y     = np.concatenate(all_y,  axis=0)
all_ts    = np.concatenate(all_ts, axis=0)
all_sites = np.array(all_sites)

print(f"X shape : {all_X.shape}  (samples, seq_len, features)")
print(f"y shape : {all_y.shape}")
print(f"NaN dans X : {np.isnan(all_X).sum()}")
print(f"NaN dans y : {np.isnan(all_y).sum()}")

## 4. Split chronologique Train / Validation / Test

In [ ]:
# On trie par timestamp de prédiction pour un split temporel propre
sort_idx = np.argsort(all_ts)
all_X, all_y, all_ts, all_sites = (
    all_X[sort_idx], all_y[sort_idx], all_ts[sort_idx], all_sites[sort_idx]
)

n = len(all_y)
n_train = int(n * 0.60)
n_val   = int(n * 0.20)

X_train, y_train = all_X[:n_train],          all_y[:n_train]
X_val,   y_val   = all_X[n_train:n_train+n_val], all_y[n_train:n_train+n_val]
X_test,  y_test  = all_X[n_train+n_val:],    all_y[n_train+n_val:]
ts_test          = all_ts[n_train+n_val:]
sites_test       = all_sites[n_train+n_val:]

print(f"Train : {X_train.shape[0]:>6} ({X_train.shape[0]/n*100:.1f}%)")
print(f"Val   : {X_val.shape[0]:>6} ({X_val.shape[0]/n*100:.1f}%)")
print(f"Test  : {X_test.shape[0]:>6} ({X_test.shape[0]/n*100:.1f}%)")

## 5. Normalisation

In [ ]:
# On fit le scaler uniquement sur les données de train (reshape 2D, puis reshape 3D)
scaler = StandardScaler()

n_train_s, seq, n_feat = X_train.shape

X_train_sc = scaler.fit_transform(X_train.reshape(-1, n_feat)).reshape(n_train_s, seq, n_feat)
X_val_sc   = scaler.transform(X_val.reshape(-1, n_feat)).reshape(X_val.shape)
X_test_sc  = scaler.transform(X_test.reshape(-1, n_feat)).reshape(X_test.shape)

print("Normalisation OK")
print(f"X_train_sc mean ≈ {X_train_sc.mean():.4f}, std ≈ {X_train_sc.std():.4f}")

## 6. Architecture LSTM

In [ ]:
def build_lstm(seq_len, n_features, lstm_units=(128, 64), dropout=0.2, lr=1e-3):
    inputs = keras.Input(shape=(seq_len, n_features), name='input')

    x = layers.LSTM(lstm_units[0], return_sequences=True, name='lstm_1')(inputs)
    x = layers.Dropout(dropout, name='drop_1')(x)

    x = layers.LSTM(lstm_units[1], return_sequences=False, name='lstm_2')(x)
    x = layers.Dropout(dropout, name='drop_2')(x)

    x = layers.Dense(32, activation='relu', name='dense_1')(x)
    output = layers.Dense(1, name='output')(x)

    model = keras.Model(inputs, output)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='mse',
        metrics=['mae']
    )
    return model


model = build_lstm(SEQ_LEN, len(FEATURE_COLS))
model.summary()

## 7. Entraînement

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1
    )
]

history = model.fit(
    X_train_sc, y_train,
    validation_data=(X_val_sc, y_val),
    epochs=100,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

## 8. Courbes d'apprentissage

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history.history['loss'],     label='Train loss')
axes[0].plot(history.history['val_loss'], label='Val loss')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history.history['mae'],     label='Train MAE')
axes[1].plot(history.history['val_mae'], label='Val MAE')
axes[1].set_title('MAE')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 9. Évaluation

In [ ]:
def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"\n{name}")
    print(f"  RMSE : {rmse:.4f}")
    print(f"  MAE  : {mae:.4f}")
    print(f"  R²   : {r2:.4f}")
    return rmse, mae, r2


y_train_pred = model.predict(X_train_sc, verbose=0).flatten()
y_val_pred   = model.predict(X_val_sc,   verbose=0).flatten()
y_test_pred  = model.predict(X_test_sc,  verbose=0).flatten()

evaluate('Train', y_train, y_train_pred)
evaluate('Validation', y_val, y_val_pred)
evaluate('Test', y_test, y_test_pred)

## 10. Visualisation des prédictions sur le test set

In [ ]:
# Affichage pour un site au choix
SITE_PLOT = sorted(np.unique(sites_test))[0]
mask = sites_test == SITE_PLOT

ts_site   = pd.to_datetime(ts_test[mask])
y_true_s  = y_test[mask]
y_pred_s  = y_test_pred[mask]

sort_t = np.argsort(ts_site)
ts_site, y_true_s, y_pred_s = ts_site[sort_t], y_true_s[sort_t], y_pred_s[sort_t]

plt.figure(figsize=(16, 4))
plt.plot(ts_site, y_true_s, label='Réel',       alpha=0.8, linewidth=1)
plt.plot(ts_site, y_pred_s, label='Prédit LSTM', alpha=0.8, linewidth=1, linestyle='--')
plt.title(f'Prédiction 24h ahead — {SITE_PLOT} (Test set)')
plt.xlabel('Date')
plt.ylabel('Utilization rate')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot prédit vs réel
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_test_pred, alpha=0.3, s=5, label='Test')
lims = [min(y_test.min(), y_test_pred.min()), max(y_test.max(), y_test_pred.max())]
plt.plot(lims, lims, 'r--', linewidth=1, label='Parfait')
plt.xlabel('Réel')
plt.ylabel('Prédit')
plt.title('Prédit vs Réel — Test set')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 11. Performances par site

In [ ]:
results = []
for site in sorted(np.unique(sites_test)):
    m = sites_test == site
    rmse = np.sqrt(mean_squared_error(y_test[m], y_test_pred[m]))
    mae  = mean_absolute_error(y_test[m], y_test_pred[m])
    r2   = r2_score(y_test[m], y_test_pred[m])
    results.append({'site': site, 'RMSE': rmse, 'MAE': mae, 'R2': r2})

df_results = pd.DataFrame(results).set_index('site')
print(df_results.to_string())
print(f"\nMoyenne — RMSE: {df_results['RMSE'].mean():.4f} | MAE: {df_results['MAE'].mean():.4f} | R²: {df_results['R2'].mean():.4f}")